In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, trim, when, to_date

spark = SparkSession.builder \
    .appName("Obligatorio Catastro - Raw to Ref") \
    .enableHiveSupport() \
    .getOrCreate()

spark.conf.set("spark.sql.parquet.datetimeRebaseModeInWrite", "CORRECTED")
spark.conf.set("spark.sql.parquet.datetimeRebaseModeInRead", "CORRECTED")

spark

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


2026-06-05T01:02:13,885 WARN [Thread-4] org.apache.hadoop.util.NativeCodeLoader - Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
RAW_PATH = "/raw/obligatorio_catastro/dnc_2026_05"
REF_PATH = "/ref/obligatorio_catastro/dnc_2026_05"

print("RAW_PATH:", RAW_PATH)
print("REF_PATH:", REF_PATH)

RAW_PATH: /raw/obligatorio_catastro/dnc_2026_05
REF_PATH: /ref/obligatorio_catastro/dnc_2026_05


In [3]:
headers = {
    "Padrones Urbanos.csv": [
        "codigo_regimen",
        "codigo_departamento",
        "codigo_localidad",
        "nro_padron",
        "block_manzana",
        "ep_ss",
        "unidad",
        "area_predio",
        "area_edificada",
        "valor_catastral_terreno",
        "valor_catastral_mejoras",
        "valor_catastral_total",
        "valor_para_impuestos",
        "fecha_ultima_djcu",
        "vigencia_ultima_djcu"
    ],
    "Padrones Rurales.csv": [
        "codigo_departamento",
        "nro_padron",
        "seccion_catastral",
        "area_predio",
        "valor_catastral_total",
        "valor_para_impuestos"
    ],
    "Departamentos.csv": [
        "codigo_departamento",
        "departamento"
    ],
    "Localidades.csv": [
        "codigo_departamento",
        "codigo_localidad",
        "localidad"
    ],
    "Destinos.csv": [
        "codigo_destino",
        "destino"
    ],
    "Categorías de Construcción.csv": [
        "codigo_categoria",
        "categoria_construccion"
    ],
    "Estados de Conservación.csv": [
        "codigo_estado",
        "estado_conservacion"
    ],
    "Cubiertas.csv": [
        "codigo_cubierta",
        "cubierta"
    ],
    "Cielorrasos.csv": [
        "codigo_cielorraso",
        "cielorraso"
    ],
    "Tipos de Obra.csv": [
        "codigo_tipo_obra",
        "tipo_obra"
    ],
    "Líneas de Construccion.csv": [
        "codigo_regimen",
        "codigo_departamento",
        "codigo_localidad",
        "nro_padron",
        "block_manzana",
        "ep_ss",
        "unidad",
        "nivel",
        "codigo_destino",
        "codigo_categoria",
        "codigo_estado",
        "codigo_cubierta",
        "codigo_cielorraso",
        "codigo_tipo_obra",
        "area_construida",
        "anio_construccion",
        "anio_remanente",
        "ep_ss_uso_exclusivo",
        "unidad_uso_exclusivo"
    ],
    "Mutaciones Catastrales.CSV": [
        "codigo_regimen",
        "codigo_departamento",
        "seccion_catastral",
        "codigo_localidad",
        "nro_padron",
        "block_manzana",
        "ep_ss",
        "unidad",
        "fecha_vigencia",
        "nro_padron_origen",
        "vigencia_padron_origen",
        "referencia"
    ],
    "Histórico de Valores.CSV": [
        "codigo_regimen",
        "codigo_departamento",
        "seccion_catastral",
        "codigo_localidad",
        "nro_padron",
        "block_manzana",
        "ep_ss",
        "unidad",
        "valor_catastral_anio_1",
        "valor_impuestos_anio_1",
        "valor_catastral_anio_2",
        "valor_impuestos_anio_2",
        "valor_catastral_anio_3",
        "valor_impuestos_anio_3",
        "valor_catastral_anio_4",
        "valor_impuestos_anio_4"
    ]
}

In [4]:
from pyspark.sql.functions import col, trim, when, to_date, regexp_extract

def normalizar_nombre_archivo(file_name):
    """
    Convierte el nombre original del archivo en un nombre válido y simple
    para guardar la tabla refinada en HDFS.
    """
    return (
        file_name
        .replace(".CSV", "")
        .replace(".csv", "")
        .lower()
        .replace(" ", "_")
        .replace("á", "a")
        .replace("é", "e")
        .replace("í", "i")
        .replace("ó", "o")
        .replace("ú", "u")
        .replace("ñ", "n")
    )


def leer_csv_sin_header(file_name, columns):
    """
    Lee un archivo CSV desde la zona RAW.
    Los archivos no tienen encabezado, por lo que luego se asignan
    los nombres de columnas definidos en los metadatos.
    """
    path = f"{RAW_PATH}/{file_name}"

    df = (
        spark.read
        .option("header", "false")
        .option("sep", ",")
        .option("quote", '"')
        .option("escape", '"')
        .option("inferSchema", "false")
        .csv(path)
    )

    if len(df.columns) != len(columns):
        raise Exception(
            f"{file_name}: esperaba {len(columns)} columnas, "
            f"pero Spark leyó {len(df.columns)}"
        )

    return df.toDF(*columns)


def trim_strings(df):
    """
    Aplica trim a todas las columnas.
    Como en esta etapa los datos se leen inicialmente como string,
    esto permite limpiar espacios al inicio y al final.
    """
    for c in df.columns:
        df = df.withColumn(c, trim(col(c)))
    return df


def limpiar_fecha(df, c):
    """
    Convierte fechas válidas en formato dd/MM/yyyy.
    Valores vacíos, '/  /', fechas mal formadas o años menores a 1900 se convierten a NULL.
    Esto evita errores de Spark al escribir fechas antiguas en Parquet.
    """
    fecha_limpia = trim(col(c))
    anio = regexp_extract(fecha_limpia, r"(\d{2})/(\d{2})/(\d{4})", 3).cast("int")

    return df.withColumn(
        c,
        when(
            (col(c).isNull()) |
            (fecha_limpia == "") |
            (fecha_limpia == "/  /") |
            (~fecha_limpia.rlike(r"^\d{2}/\d{2}/\d{4}$")) |
            (anio < 1900),
            None
        ).otherwise(to_date(fecha_limpia, "dd/MM/yyyy"))
    )

In [5]:
procesadas = []
errores = []

for file_name, cols in headers.items():
    try:
        print(f"Procesando: {file_name}")

        df = leer_csv_sin_header(file_name, cols)
        df = trim_strings(df)

        if file_name == "Padrones Urbanos.csv":
            df = limpiar_fecha(df, "fecha_ultima_djcu")
            df = limpiar_fecha(df, "vigencia_ultima_djcu")

        if file_name == "Mutaciones Catastrales.CSV":
            df = limpiar_fecha(df, "fecha_vigencia")

        output_name = normalizar_nombre_archivo(file_name)
        output_path = f"{REF_PATH}/{output_name}"

        (
            df.write
            .mode("overwrite")
            .parquet(output_path)
        )

        procesadas.append(output_name)
        print(f"OK: {output_path}")

    except Exception as e:
        errores.append((file_name, str(e)))
        print(f"ERROR en {file_name}: {e}")

print("\nTablas procesadas correctamente:")
for t in procesadas:
    print("-", t)

print("\nErrores:")
if len(errores) == 0:
    print("No se encontraron errores.")
else:
    for file_name, error in errores:
        print(file_name, "=>", error)

Procesando: Padrones Urbanos.csv


OK: /ref/obligatorio_catastro/dnc_2026_05/padrones_urbanos
Procesando: Padrones Rurales.csv


OK: /ref/obligatorio_catastro/dnc_2026_05/padrones_rurales
Procesando: Departamentos.csv
OK: /ref/obligatorio_catastro/dnc_2026_05/departamentos
Procesando: Localidades.csv
OK: /ref/obligatorio_catastro/dnc_2026_05/localidades
Procesando: Destinos.csv
OK: /ref/obligatorio_catastro/dnc_2026_05/destinos
Procesando: Categorías de Construcción.csv
OK: /ref/obligatorio_catastro/dnc_2026_05/categorias_de_construccion
Procesando: Estados de Conservación.csv
OK: /ref/obligatorio_catastro/dnc_2026_05/estados_de_conservacion
Procesando: Cubiertas.csv
OK: /ref/obligatorio_catastro/dnc_2026_05/cubiertas
Procesando: Cielorrasos.csv


OK: /ref/obligatorio_catastro/dnc_2026_05/cielorrasos
Procesando: Tipos de Obra.csv
OK: /ref/obligatorio_catastro/dnc_2026_05/tipos_de_obra
Procesando: Líneas de Construccion.csv


OK: /ref/obligatorio_catastro/dnc_2026_05/lineas_de_construccion
Procesando: Mutaciones Catastrales.CSV
OK: /ref/obligatorio_catastro/dnc_2026_05/mutaciones_catastrales
Procesando: Histórico de Valores.CSV


[Stage 25:=============================>                            (1 + 1) / 2]

OK: /ref/obligatorio_catastro/dnc_2026_05/historico_de_valores

Tablas procesadas correctamente:
- padrones_urbanos
- padrones_rurales
- departamentos
- localidades
- destinos
- categorias_de_construccion
- estados_de_conservacion
- cubiertas
- cielorrasos
- tipos_de_obra
- lineas_de_construccion
- mutaciones_catastrales
- historico_de_valores

Errores:
No se encontraron errores.


In [6]:
print("Cantidad de tablas esperadas:", len(headers))
print("Cantidad de tablas procesadas:", len(procesadas))

if len(errores) > 0:
    raise Exception("Existen errores en el procesamiento. Revisar la salida anterior.")

if len(procesadas) != len(headers):
    raise Exception("No se procesaron todas las tablas esperadas.")

print("Validación OK: todas las tablas fueron procesadas.")

Cantidad de tablas esperadas: 13
Cantidad de tablas procesadas: 13
Validación OK: todas las tablas fueron procesadas.


In [7]:
df_departamentos = spark.read.parquet(f"{REF_PATH}/departamentos")
df_padrones_urbanos = spark.read.parquet(f"{REF_PATH}/padrones_urbanos")
df_lineas_construccion = spark.read.parquet(f"{REF_PATH}/lineas_de_construccion")

df_departamentos.show(5, truncate=False)

print("Cantidad de departamentos:", df_departamentos.count())
print("Cantidad de padrones urbanos:", df_padrones_urbanos.count())
print("Cantidad de líneas de construcción:", df_lineas_construccion.count())

+-------------------+--------------+
|codigo_departamento|departamento  |
+-------------------+--------------+
|A                  |CANELONES     |
|B                  |MALDONADO     |
|C                  |ROCHA         |
|D                  |TREINTA Y TRES|
|E                  |CERRO LARGO   |
+-------------------+--------------+
only showing top 5 rows

Cantidad de departamentos: 19
Cantidad de padrones urbanos: 1468837
Cantidad de líneas de construcción: 4478065
